The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

1\.d Find out which is the letter with most counts.

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [7]:
codice = """
#! /bin/bash

##ho creato una cartella 'students' 
mkdir students 

##sono entrata nella cartella 'students'
cd students

##ho creato un file csv_file
touch csv_file 

##cerco se esiste il file da cui voglio scaricare i dati nella cartella, eventualmente lo creo e poi copio i suoi dati su csv_file

if [ -f "LCP_22-23_students.csv"]; then
    echo "File already exists locally."
    cp ../LCP_22-23_students.csv csv_file
else
    wget LCP_22-23_students.csv "https://www.dropbox.com/scl/fi/bxv17nrbrl83vw6qrkiu9/LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7&e=1"
    cp LCP_22-23_students.csv csv_file  
fi

#creato i file nuovi nella cartella
touch Physics_students
touch PoD_students

##copiato la riga di intestazione
head -n 1 csv_file > Physics_students
head -n 1 csv_file > PoD_students|

##aggiungo i corrispettivi studenti 
grep 'PoD' csv_file>>PoD_Students
grep 'Physics' csv_file >> Physics_Students

#For each letter of the alphabet, count the number of students whose surname starts with that letter.

mostcount=0
letter='a'

for i in {A..Z}; do
    count=$(tail -n +2 csv_file | awk '{print $2}' | grep -c "^$i")
    if [ "$count" -gt "$mostcount" ]; then
        mostcount=$count
        letter="$i"
    fi
    echo "Surnames starting with $i: $count"
done
echo "Surnames starting with $letter are the most numerous, they are $mostcount"

#creo 18 file con intestazione uguale a csv_file, ogni file è chiamato con il resto della divisione tra ilo numero dello studente e 18, quindi gli studenti (1,19,37,...)=gruppo1, (2,20,38,...)=gruppo2, (18,36,...)=gruppo0
for i in {0..17}; do
    head -n 1 csv_file > "group_$i.csv"
fi

student_num=1
tail -n +2 csv_file | while IFS= read -r line; do
    resto=$((student_num % 18))
    echo "$line" >> "group_$resto.csv"
    
    student_num=$((student_num + 1)
done
"""

with open("03_ex1.sh", "w") as f:
    f.write(codice)


after running the cell above, open terminal and write:
```bash
chmod +x 03_ex1.sh
./03_ex1.sh 

### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

2\.b How many even numbers are there?

c=0
while IFS= read -r line; do
    read -r -a numbers <<< "$line"
    for number in "${numbers[@]}"; do
        if ((number % 2 == 0)); then (( c ++ )); fi
    done
done < 'data.txt'
echo "there are $c even numbers"

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

greater=0
smaller=0
value=$(echo "scale=2; 50*sqrt(3)"|bc -l)
while IFS= read -r line; do
    read -r -a numbers <<< "$line"
    x1=${numbers[0]}
    y1=${numbers[1]}
    z1=${numbers[2]}
    x2=${numbers[3]}
    y2=${numbers[4]}
    z2=${numbers[5]}
    distance1=$(echo "scale=6; sqrt($x1*$x1 + $y1*$y1 +$z1*$z1)" | bc -l)
    distance2=$(echo "scale=6; sqrt($x2*$x2 + $y2*$y2 +$z2*$z2)" | bc -l)
    if [ "$(echo "$distance1 < $value" | bc -l)" -eq 1 ]; then
        ((smaller ++))
    else
        ((greater ++))
    fi
    if [ "$(echo "$distance2 < $value" | bc -l)" -eq 1 ]; then
        ((smaller ++))
    else
        ((greater ++))
    fi
done < 'data.txt'
echo "there are $greater groups which are greater than $value"
echo "there are $smaller groups which are smaller than $value"

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

read -p "Inserisci il numero di copie da generare (n): " n
for ((i=1; i<=n; i++)); do
    output_file="data_${i}.txt"
    cp data.txt "$output_file"

    temp_file="temp.txt"
    > "$temp_file"
    
    while IFS= read -r line; do
        read -r -a numbers <<< "$line"
        new_line=""
        for number in "${numbers[@]}"; do
            divided=$(echo "scale=2; $number / $i" | bc -l)
            new_line="$new_line $divided"
        done
        echo "${new_line# }" >> "$temp_file"
    done < "$output_file"
    mv "$temp_file" "$output_file"
done 
    
            

    

In [ ]:
codice='''
#! /bin/bash

#per prima cosa ho creato il file nella cartella:

touch data.csv

poi ho copiato i dati della lezione precedente 
echo "# Recorded data from sensor XaB.v2
# Recording started on Oct 29 15:00:00
# Recording ended on Oct 29 15:15:59
# X, Y, Z, X', Y', Z'

the ashtag are used as keys to interpret tha data below

55, 14, 48, 62, 78, 41
32, 52, 94, 11, 17, 83
83, 13, 61, 69, 44, 39
77, 34, 34, 2, 93, 24
25, 28, 49, 92, 3, 8
44, 87, 63, 62, 23, 98
90, 20, 6, 79, 32, 30
73, 91, 41, 36, 0, 93
63, 96, 17, 56, 30, 81
37, 56, 35, 90, 69, 35
32, 54, 87, 61, 81, 99
21, 77, 47, 54, 99, 36
93, 7, 85, 12, 67, 16
65, 53, 27, 59, 98, 94
23, 7, 59, 19, 75, 13
47, 68, 79, 83, 81, 86
51, 49, 84, 96, 71, 94
23, 99, 9, 17, 55, 70
49, 79, 62, 51, 6, 96
56, 16, 51, 18, 11, 23
98, 0, 78, 80, 75, 53
77, 30, 58, 41, 86, 17
93, 40, 62, 37, 92, 20
30, 86, 63, 45, 70, 99
2, 63, 89, 41, 95, 84
77, 10, 87, 24, 2, 86
43, 17, 82, 21, 52, 12
33, 65, 98, 38, 19, 10
52, 17, 68, 26, 61, 5
39, 94, 40, 99, 48, 20
58, 79, 79, 69, 71, 64
6, 12, 97, 4, 16, 98
79, 0, 40, 83, 49, 65
51, 13, 29, 84, 62, 29
60, 34, 12, 1, 12, 32
36, 92, 15, 44, 82, 60
6, 10, 72, 22, 62, 62
69, 98, 3, 34, 16, 61
21, 25, 2, 24, 93, 45
82, 18, 2, 79, 51, 73
93, 57, 75, 52, 77, 27
10, 61, 93, 79, 74, 67
6, 45, 94, 67, 68, 63
2, 54, 8, 32, 86, 49
60, 52, 22, 1, 19, 73
78, 70, 87, 87, 39, 86
0, 66, 89, 61, 28, 42
85, 36, 81, 45, 49, 80
96, 31, 82, 32, 88, 15
70, 95, 3, 53, 31, 48" > data.csv

#adesso creo una copia e rimuovo metadata e virgole

grep -Ev '^(#|[Tt]he)' data.csv | tr -d ',' > data.txt

#How many even numbers are there?
c=0
while IFS= read -r line; do
    read -r -a numbers <<< "$line"
    for number in "${numbers[@]}"; do
        if ((number % 2 == 0)); then (( c ++ )); fi
    done
done < 'data.txt'
echo "there are $c even numbers"

greater=0
smaller=0
value=$(echo "scale=2; 50*sqrt(3)"|bc -l)
while IFS= read -r line; do
    read -r -a numbers <<< "$line"
    x1=${numbers[0]}
    y1=${numbers[1]}
    z1=${numbers[2]}
    x2=${numbers[3]}
    y2=${numbers[4]}
    z2=${numbers[5]}
    distance1=$(echo "scale=6; sqrt($x1*$x1 + $y1*$y1 +$z1*$z1)" | bc -l)
    distance2=$(echo "scale=6; sqrt($x2*$x2 + $y2*$y2 +$z2*$z2)" | bc -l)
    if [ "$(echo "$distance1 < $value" | bc -l)" -eq 1 ]; then
        ((smaller ++))
    else
        ((greater ++))
    fi
    if [ "$(echo "$distance2 < $value" | bc -l)" -eq 1 ]; then
        ((smaller ++))
    else
        ((greater ++))
    fi
done < 'data.txt'
echo "there are $greater groups which are greater than $value"
echo "there are $smaller groups which are smaller than $value"

read -p "Inserisci il numero di copie da generare (n): " n
for ((i=1; i<=n; i++)); do
    output_file="data_${i}.txt"
    cp data.txt "$output_file"

    temp_file="temp.txt"
    > "$temp_file"
    
    while IFS= read -r line; do
        read -r -a numbers <<< "$line"
        new_line=""
        for number in "${numbers[@]}"; do
            divided=$(echo "scale=2; $number / $i" | bc -l)
            new_line="$new_line $divided"
        done
        echo "${new_line# }" >> "$temp_file"
    done < "$output_file"
    mv "$temp_file" "$output_file"
done 
'''

with open("03_ex2.sh", "w") as f:
    f.write(codice)


after running the cell above, open terminal and write:

    chmod +x 03_ex2.sh
    ./03_ex2.sh 